# 阶段五：取消传播

模拟用户停止请求，观察取消从批次传递到具体工具，并确保 `finally` 执行。

In [ ]:
import asyncio

async def cancellable_tool(name: str, delay: float, cleaned: list[str]) -> str:
    """模拟可取消的查询工具。

    参数:
        name: 工具名称。
        delay: 每次等待的秒数。
        cleaned: 记录清理完成的列表。

    返回:
        工具成功完成时的结果。
    """
    try:
        await asyncio.sleep(delay)
        return f"{name}: done"
    except asyncio.CancelledError:
        print(f"{name}: cancelled")
        raise
    finally:
        cleaned.append(name)
        print(f"{name}: cleanup")

async def run_cancellable_batch(names: list[str], delay: float) -> list[str]:
    """并发运行查询工具，并在父任务取消时传播取消。

    参数:
        names: 工具名称列表。
        delay: 每个工具的等待秒数。

    返回:
        所有工具成功时的结果列表。
    """
    cleaned: list[str] = []
    try:
        return await asyncio.gather(*(cancellable_tool(name, delay, cleaned) for name in names))
    finally:
        print(f"cleaned {len(cleaned)} tool(s)")

async def native_cancel_example() -> None:
    """使用原生 `Task.cancel()` 取消整个工具批次。"""
    task = asyncio.create_task(run_cancellable_batch(["search", "weather"], 1.0))
    await asyncio.sleep(0.05)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("parent cancellation propagated")

async def langchain_async_example() -> None:
    """通过 LangChain Runnable 的 `ainvoke` 运行可取消工具。"""
    from langchain_core.runnables import RunnableLambda
    cleaned: list[str] = []
    runnable = RunnableLambda(func=lambda name: name, afunc=lambda name: cancellable_tool(name, 1.0, cleaned))
    task = asyncio.create_task(runnable.ainvoke("search"))
    await asyncio.sleep(0.05)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        assert cleaned == ["search"]
        print("LangChain Async cancellation propagated")

async def main() -> None:
    """运行原生 Python 和 LangChain Async 的取消验收。"""
    await native_cancel_example()
    await langchain_async_example()

await main()


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.post("/phase5/cancel")
async def run_phase5_cancel() -> dict[str, str]:
    """提供 HTTP 入口，运行并取消一个异步批次。

    返回:
        批次被取消后的状态。
    """
    task = asyncio.create_task(run_cancellable_batch(["search", "weather"], 1.0))
    await asyncio.sleep(0.05)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        return {"status": "cancelled"}
    return {"status": "completed"}

assert any(route.path == "/phase5/cancel" for route in app.routes)
print("FastAPI route registered")
